In [1]:
from google.cloud import bigquery

client = bigquery.Client()
print(client.project)

poai-research


In [ ]:
# result = client.query("SELECT 1 AS test").to_dataframe()
# print(result)

   test
0     1


In [ ]:
# job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# query = """
# SELECT
#   block_number,
#   block_timestamp,
#   ARRAY(
#     SELECT AS STRUCT
#       addr AS address,
#       out.type AS script_type,
#       out.value AS value
#     FROM UNNEST(outputs) AS out, UNNEST(out.addresses) AS addr
#     WHERE out.type != 'nonstandard'
#   ) AS real_outputs
# FROM `bigquery-public-data.crypto_bitcoin.transactions`
# WHERE is_coinbase = true
#   AND block_number BETWEEN 0 AND 810908
#   AND block_timestamp_month BETWEEN '2009-01-01' AND '2023-10-01'
# """

# dry_run_job = client.query(query, job_config=job_config)
# print(f"This query will process {dry_run_job.total_bytes_processed / 1e9:.2f} GB")

This query will process 197.54 GB


In [ ]:
# query = """
# SELECT
#   block_number,
#   block_timestamp,
#   ARRAY(
#     SELECT AS STRUCT
#       addr AS address,
#       out.type AS script_type,
#       out.value AS value
#     FROM UNNEST(outputs) AS out, UNNEST(out.addresses) AS addr
#     WHERE out.type != 'nonstandard'
#   ) AS real_outputs
# FROM `bigquery-public-data.crypto_bitcoin.transactions`
# WHERE is_coinbase = true
#   AND block_number BETWEEN 0 AND 810908
#   AND block_timestamp_month BETWEEN '2009-01-01' AND '2023-10-01'
# """

# df = client.query(query).to_dataframe()
# df.to_csv("../data/coinbase_addresses_full.csv", index=False)
# print(f"Pulled {len(df)} rows")

Pulled 810909 rows


In [4]:
df

,block_number,block_timestamp,real_outputs
0,755688,2022-09-25 21:17:12+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...
1,755658,2022-09-25 15:04:25+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...
2,756371,2022-09-30 13:14:40+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...
3,754843,2022-09-19 20:29:55+00:00,[{'address': '3L8Ck6bm3sve1vJGKo6Ht2k167YKSKi8...
4,755220,2022-09-22 11:21:06+00:00,[{'address': '3C9sAKXrBVpJVe3b738yik4LPHpPmceB...
...,...,...,...
810904,762082,2022-11-07 04:51:06+00:00,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...
810905,761961,2022-11-06 09:03:17+00:00,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...
810906,763253,2022-11-15 08:27:13+00:00,[{'address': '12KKDt4Mj7N5UAkQMN7LtPZMayenXHa8...
810907,762893,2022-11-12 15:41:42+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...


In [5]:
print(df.shape)          # should be (810909, 3)
print(df.dtypes)
df.head(5)

(810909, 3)
block_number                     Int64
block_timestamp    datetime64[us, UTC]
real_outputs                    object
dtype: object


,block_number,block_timestamp,real_outputs
0,755688,2022-09-25 21:17:12+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...
1,755658,2022-09-25 15:04:25+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...
2,756371,2022-09-30 13:14:40+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...
3,754843,2022-09-19 20:29:55+00:00,[{'address': '3L8Ck6bm3sve1vJGKo6Ht2k167YKSKi8...
4,755220,2022-09-22 11:21:06+00:00,[{'address': '3C9sAKXrBVpJVe3b738yik4LPHpPmceB...


In [6]:
print(df.iloc[0]["real_outputs"])
print(df.iloc[-1]["real_outputs"])

[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX', 'script_type': 'pubkeyhash', 'value': Decimal('627599947.000000000')}]
[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX', 'script_type': 'pubkeyhash', 'value': Decimal('652797442.000000000')}]


In [7]:
df["n_addresses"] = df["real_outputs"].apply(len)
df["n_addresses"].value_counts()

n_addresses
1      776074
2       19435
21       1046
5         635
3         334
        ...  
438         1
481         1
493         1
394         1
504         1
Name: count, Length: 511, dtype: int64

In [8]:
df_sorted_asc = df.sort_values("block_number", ascending=True)
df_sorted_asc.head(10)

,block_number,block_timestamp,real_outputs,n_addresses
695788,0,2009-01-03 18:15:05+00:00,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,1
693439,1,2009-01-09 02:54:25+00:00,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,1
693519,2,2009-01-09 02:55:44+00:00,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,1
693268,3,2009-01-09 03:02:53+00:00,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,1
693338,4,2009-01-09 03:16:28+00:00,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,1
693802,5,2009-01-09 03:23:48+00:00,[{'address': '1JfbZRwdDHKZmuiZgYArJZhcuuzuw2Hu...,1
693824,6,2009-01-09 03:29:49+00:00,[{'address': '1GkQmKAmHtNfnD3LHhTkewJxKHVSta4m...,1
693425,7,2009-01-09 03:39:29+00:00,[{'address': '16LoW7y83wtawMg5XmT4M3Q7EdjjUmen...,1
693403,8,2009-01-09 03:45:43+00:00,[{'address': '1J6PYEzr4CUoGbnXrELyHszoTSz3wCsC...,1
693776,9,2009-01-09 03:54:39+00:00,[{'address': '12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu...,1


In [9]:
df_sorted_desc = df.sort_values("block_number", ascending=False)
df_sorted_desc.head(10)

,block_number,block_timestamp,real_outputs,n_addresses
515788,810908,2023-10-06 13:37:21+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1
509392,810907,2023-10-06 13:37:00+00:00,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,1
593194,810906,2023-10-06 12:50:15+00:00,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,2
509400,810905,2023-10-06 12:45:49+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1
472630,810904,2023-10-06 12:36:45+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1
554216,810903,2023-10-06 12:27:41+00:00,[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5...,1
554215,810902,2023-10-06 12:15:02+00:00,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,1
472631,810901,2023-10-06 12:01:05+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1
620242,810900,2023-10-06 11:54:39+00:00,[{'address': 'bc1qnxqs46jyvdf2elelsg96g8xezttc...,2
510763,810899,2023-10-06 11:54:09+00:00,[{'address': '3L8Ck6bm3sve1vJGKo6Ht2k167YKSKi8...,1


In [10]:
print(df[df["block_number"] == 0]["real_outputs"].values)

[array([{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa', 'script_type': 'pubkey', 'value': Decimal('5000000000.000000000')}],
       dtype=object)                                                                                                          ]


In [11]:
df["n_addresses"].value_counts()

n_addresses
1      776074
2       19435
21       1046
5         635
3         334
        ...  
438         1
481         1
493         1
394         1
504         1
Name: count, Length: 511, dtype: int64

In [12]:
multi = df[df["n_addresses"] >= 20].iloc[0]
print(multi["block_number"], multi["block_timestamp"])
for out in multi["real_outputs"][:10]:
    print(out)

228701 2013-03-30 04:20:08+00:00
{'address': '1LXGFUwo25C2piWNuXWVR1UZMLpw9kEVdS', 'script_type': 'pubkeyhash', 'value': Decimal('4120.000000000')}
{'address': '1BW1aQNT3VcQ6LLXX9jjKv75PVmK2PTj16', 'script_type': 'pubkeyhash', 'value': Decimal('602550.000000000')}
{'address': '13SXMpXHiuLyTqL7D7m6uKdiinkbFxwng1', 'script_type': 'pubkeyhash', 'value': Decimal('429510.000000000')}
{'address': '1unjnHrqjyfvKee1GgYRksnVHsR8iHvKm', 'script_type': 'pubkeyhash', 'value': Decimal('9107344.000000000')}
{'address': '1AMcEK2k5c72azXYS4hSJYHiqNHPXeeZXv', 'script_type': 'pubkeyhash', 'value': Decimal('2578426.000000000')}
{'address': '18ZjUoRzMYz4Cg4XzLB5oLyjMthEhnNoCG', 'script_type': 'pubkeyhash', 'value': Decimal('48925.000000000')}
{'address': '1K8ytxoaXMmwBcjoNgaq1YxRiLXXHnGGbe', 'script_type': 'pubkeyhash', 'value': Decimal('67980.000000000')}
{'address': '1C3bTEPV33FsAkjZS1vBT5QFe4u6E4wRUR', 'script_type': 'pubkeyhash', 'value': Decimal('1006310.000000000')}
{'address': '1Ps4YJpHhQ4ZBBrnGy6w

In [15]:
print(repr(df["real_outputs"].iloc[0]))

"[{'address': '18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX', 'script_type': 'pubkeyhash', 'value': Decimal('627599947.000000000')}]"


In [18]:
from decimal import Decimal

def parse_outputs(s):
    return eval(s, {"Decimal": Decimal, "__builtins__": {}})

for idx, s in df["real_outputs"].items():
    try:
        parse_outputs(s)
    except SyntaxError as e:
        print(f"Failed at row index: {idx}")
        print(f"block_number: {df.loc[idx, 'block_number']}")
        print(f"String length: {len(s)}")
        print(f"Raw repr:\n{s!r}")
        break

Failed at row index: 24
block_number: 419094
String length: 242
Raw repr:
"[{'address': '13pucx6gHP2vyBLc88QfcGivjkhK63PeVg', 'script_type': 'pubkeyhash', 'value': Decimal('2509616020.000000000')}\n {'address': '19ZKM6JFvCiBQbqqHPzRDLGHpN6wkQnXDs', 'script_type': 'pubkeyhash', 'value': Decimal('22791669.000000000')}]"


In [20]:
import pandas as pd
import re
from decimal import Decimal

df = pd.read_csv("../data/coinbase_addresses_full.csv")

# 1. Row count check
print("Row count:", len(df))
print("Row count == 810909?", len(df) == 810909)

# Fix: the BigQuery client returned nested fields as numpy arrays. When pandas
# wrote them to CSV, numpy's str() formatting separates dict entries with a
# newline+space instead of a comma — valid for numpy's own printing, but not
# valid Python list syntax. This regex restores the comma between entries
# before we hand the string to eval().
def parse_outputs(s):
    fixed = re.sub(r"\}\s*\n\s*\{", "}, {", s)
    return eval(fixed, {"Decimal": Decimal, "__builtins__": {}})

df["real_outputs_parsed"] = df["real_outputs"].apply(parse_outputs)
df["n_addresses"] = df["real_outputs_parsed"].apply(len)

# 2. Genesis block check
genesis = df[df["block_number"] == 0].iloc[0]
print("\nGenesis block outputs:", genesis["real_outputs_parsed"])

# 3. Tip block check
tip = df[df["block_number"] == 810908].iloc[0]
print("\nTip block timestamp:", tip["block_timestamp"])

# 4. Single-address block
single = df[df["n_addresses"] == 1].iloc[100]
print("\nExample single-address block:")
print("  block_number:", single["block_number"])
print("  timestamp:", single["block_timestamp"])
print("  outputs:", single["real_outputs_parsed"])

# 5. Multi-address block
multi = df[df["n_addresses"] == 2].iloc[0]
print("\nExample 2-address block:")
print("  block_number:", multi["block_number"])
print("  outputs:", multi["real_outputs_parsed"])

# 6. Worth adding: confirm the fix didn't silently fail anywhere else.
# If this prints anything, something is still wrong on that row specifically.
bad_rows = []
for idx, s in df["real_outputs"].items():
    try:
        parse_outputs(s)
    except Exception as e:
        bad_rows.append((idx, str(e)))
print(f"\nRows that still fail to parse: {len(bad_rows)}")
if bad_rows[:5]:
    print(bad_rows[:5])

Row count: 810909
Row count == 810909? True

Genesis block outputs: [{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa', 'script_type': 'pubkey', 'value': Decimal('5000000000.000000000')}]

Tip block timestamp: 2023-10-06 13:37:21+00:00

Example single-address block:
  block_number: 224245
  timestamp: 2013-03-04 15:50:11+00:00
  outputs: [{'address': '19fbcmiSmxHGjyBDG2on5Qgfnvxs8ukdNz', 'script_type': 'pubkeyhash', 'value': Decimal('2512255000.000000000')}]

Example 2-address block:
  block_number: 419094
  outputs: [{'address': '13pucx6gHP2vyBLc88QfcGivjkhK63PeVg', 'script_type': 'pubkeyhash', 'value': Decimal('2509616020.000000000')}, {'address': '19ZKM6JFvCiBQbqqHPzRDLGHpN6wkQnXDs', 'script_type': 'pubkeyhash', 'value': Decimal('22791669.000000000')}]

Rows that still fail to parse: 0
